# Clustering Analysis — Product Category Segmentation

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

In [ ]:
with open(os.path.join(DATA_DIR, 'product-category-benchmarks.json')) as f:
    benchmarks = json.load(f)

rows = []
for cat, data in benchmarks['categories'].items():
    rows.append({
        'category': cat,
        'co2e_median': data['co2eKg']['median'],
        'water_median': data['waterLiters']['median'],
        'energy_median': data['energyKwh']['median'],
        'price_median': data['typicalPrice']['median'],
        'lifespan_years': data['typicalLifespan']['years'],
        'co2e_range': data['co2eKg']['max'] - data['co2eKg']['min'],
    })

df = pd.DataFrame(rows).set_index('category')
feature_cols = ['co2e_median', 'water_median', 'energy_median', 'price_median', 'lifespan_years', 'co2e_range']

scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])
print(f'Feature matrix shape: {X.shape}')
df.head()

## Silhouette Analysis

In [ ]:
k_range = range(2, min(11, X.shape[0]))
silhouette_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    silhouette_scores.append(silhouette_score(X, labels))
    inertias.append(km.inertia_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(k_range), silhouette_scores, 'o-', color='#2ecc71', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Silhouette Score')
ax1.set_title('Silhouette Score vs. k')
ax1.grid(True, alpha=0.3)

ax2.plot(list(k_range), inertias, 'o-', color='#e74c3c', linewidth=2)
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Inertia')
ax2.set_title('Elbow Plot')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/silhouette_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f'Best k by silhouette score: {best_k} (score={max(silhouette_scores):.3f})')

## PCA Dimensionality Reduction

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 5))
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax.bar(range(1, len(cumvar) + 1), pca.explained_variance_ratio_, alpha=0.6, color='#3498db', label='Individual')
ax.plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='#e74c3c', linewidth=2, label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Explained Variance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(X)
df['cluster'] = cluster_labels

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='Set2',
                     s=100, edgecolors='white', linewidth=0.5)

for i, cat in enumerate(df.index):
    ax.annotate(cat, (X_pca[i, 0], X_pca[i, 1]),
                fontsize=7, ha='center', va='bottom', alpha=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title(f'PCA — Product Categories (k={best_k})')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('../figures/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## t-SNE Visualization

In [ ]:
perplexity = min(5, X.shape[0] - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
X_tsne = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='Set2',
                     s=120, edgecolors='white', linewidth=0.5)

for i, cat in enumerate(df.index):
    ax.annotate(cat, (X_tsne[i, 0], X_tsne[i, 1]),
                fontsize=7, ha='center', va='bottom', alpha=0.8)

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE — Product Category Clusters')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('../figures/tsne_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## Cluster Interpretation

Examine the mean feature values per cluster to understand what each segment represents.
High-impact electronics clusters tend to separate from lightweight consumer goods and textiles.